# SPK AXA Insurance — Sistem Pendukung Keputusan Klaim
## Framework: CRISP-DM | Fase: Modeling, Evaluation, dan Decision Output

**Konteks Bisnis:** Kenaikan klaim kesehatan sebesar 25.5 persen YoY mengancam stabilitas finansial.
Notebook ini mengimplementasikan Hybrid AI (Core Models) dikombinasikan dengan Sistem Pakar Tradisional
(Comparator Models) untuk mengidentifikasi nasabah berisiko tinggi secara presisi.

**Arsitektur Model:**
- **Core Models:** Isolation Forest (Anomaly Detection) | K-Means (Risk Clustering) | Regression (Cost Prediction)
- **Comparator Models:** Forward/Backward Chaining | Teorema Bayes | Certainty Factors
- **Decision Engine:** Dynamic Percentile Ranking untuk Top 5% Nasabah Kritis

> **Cara Penggunaan di Google Colab:** Upload file `AXA_Prepared_Data.csv` ke sesi Colab Anda,
> lalu jalankan seluruh sel secara berurutan (Runtime > Run All).


## Sel 1 — Instalasi dan Impor Library

Mengimpor seluruh dependensi yang diperlukan. Pastikan semua library tersedia sebelum melanjutkan ke sel berikutnya.

In [ ]:
# ============================================================
# SEL 1: INSTALASI DAN IMPOR LIBRARY
# Semua library yang digunakan adalah bagian dari ekosistem
# standar data science Python. Tidak ada dependensi eksternal
# yang memerlukan instalasi tambahan di Google Colab.
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings

from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    silhouette_score, accuracy_score, classification_report
)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# Konfigurasi tampilan global
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", 50)
plt.rcParams.update({
    "figure.dpi"      : 120,
    "axes.spines.top" : False,
    "axes.spines.right": False,
    "font.family"     : "DejaVu Sans",
})

# Palet warna AXA (merah korporat + abu-abu)
AXA_RED    = "#FF1721"
AXA_DARK   = "#1A1A2E"
AXA_GRAY   = "#6C757D"
AXA_LIGHT  = "#F8F9FA"
AXA_BLUE   = "#0D6EFD"
AXA_GREEN  = "#198754"

print("Library berhasil diimpor.")
print(f"  pandas     : {pd.__version__}")
print(f"  numpy      : {np.__version__}")
print(f"  sklearn    : {__import__('sklearn').__version__}")


## Sel 2 — Load dan Inspeksi Data Master

Memuat `AXA_Prepared_Data.csv` yang sudah bersih dan feature-rich dari tahap Data Preparation. Seluruh proses modeling bergantung pada file ini sebagai satu-satunya sumber data.

In [ ]:
# ============================================================
# SEL 2: LOAD DATA MASTER
# File AXA_Prepared_Data.csv adalah output dari pipeline
# CRISP-DM fase Data Preparation. File ini sudah bersih,
# ter-merge, dan memiliki 44 kolom fitur siap pakai.
# ============================================================

# Untuk Google Colab: sesuaikan path ini jika perlu
# Contoh jika upload manual: df = pd.read_csv('AXA_Prepared_Data.csv')

DATA_PATH = "AXA_Prepared_Data.csv"

df = pd.read_csv(DATA_PATH)

print(f"Data berhasil dimuat: {df.shape[0]:,} baris x {df.shape[1]} kolom")
print()

# ---- Identifikasi kolom berdasarkan tipe dan peran ----
SCALED_FEATURES = [c for c in df.columns if c.endswith("_Scaled")]
ENCODED_FEATURES = [c for c in df.columns if c.endswith("_Encoded")]
FLAG_FEATURES = [c for c in df.columns if c.endswith("_Flag")]

# Fitur utama untuk Core Models (sudah dalam skala [0,1])
CORE_FEATURES = SCALED_FEATURES + ["Lokasi_RS_Risk_Score",
                                    "Is_Inpatient",
                                    "Quarter_Claim",
                                    "Month_Claim"]

TARGET_COL = "Nominal Klaim Yang Disetujui"

print(f"Jumlah Scaled Features  : {len(SCALED_FEATURES)}")
print(f"Jumlah Encoded Features : {len(ENCODED_FEATURES)}")
print(f"Jumlah Flag Features    : {len(FLAG_FEATURES)}")
print(f"Total Core Features     : {len(CORE_FEATURES)}")
print()
print("Ringkasan statistik fitur keuangan utama:")
print(df[[TARGET_COL, "Nominal Biaya RS Yang Terjadi", "Cost_Gap"]].describe())


## Sel 3 — Eksplorasi Data (EDA) Kontekstual

Visualisasi eksploratif untuk membangun intuisi tentang distribusi klaim sebelum modeling. Ini adalah langkah kritis agar kita tidak membangun model di atas asumsi yang salah.

In [ ]:
# ============================================================
# SEL 3: EXPLORATORY DATA ANALYSIS
# Tiga pertanyaan bisnis kritis yang divisualisasikan:
# (a) Bagaimana distribusi nominal klaim? (untuk justifikasi log-transform)
# (b) Bagaimana tren klaim per bulan? (untuk konteks 25.5% YoY)
# (c) Apa komposisi tipe perawatan dan lokasi RS?
# ============================================================

fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# ---- Plot 1: Distribusi Nominal Klaim (raw vs log) ----
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(df[TARGET_COL].clip(upper=df[TARGET_COL].quantile(0.99)),
         bins=50, color=AXA_RED, alpha=0.8, edgecolor="white")
ax1.set_title("Distribusi Nominal Klaim (P99 clip)", fontweight="bold")
ax1.set_xlabel("Nominal Klaim (IDR)")
ax1.set_ylabel("Frekuensi")
ax1.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))

ax2 = fig.add_subplot(gs[0, 1])
log_klaim = np.log1p(df[TARGET_COL].clip(lower=1))
ax2.hist(log_klaim, bins=50, color=AXA_DARK, alpha=0.8, edgecolor="white")
ax2.set_title("Distribusi Log(Nominal Klaim + 1)", fontweight="bold")
ax2.set_xlabel("Log Nominal Klaim")
ax2.set_ylabel("Frekuensi")

# ---- Plot 2: Tren Klaim Per Bulan ----
ax3 = fig.add_subplot(gs[0, 2])
monthly = df.groupby(["Year_Claim", "Month_Claim"])[TARGET_COL].agg(
    ["count", "sum"]).reset_index()
monthly["label"] = monthly["Year_Claim"].astype(str) + "-M" +                    monthly["Month_Claim"].astype(str).str.zfill(2)

for year, color, label in [(2024, AXA_BLUE, "2024"), (2025, AXA_RED, "2025")]:
    sub = monthly[monthly["Year_Claim"] == year]
    ax3.plot(sub["Month_Claim"], sub["count"],
             marker="o", color=color, label=label, linewidth=2)

ax3.set_title("Tren Volume Klaim per Bulan", fontweight="bold")
ax3.set_xlabel("Bulan")
ax3.set_ylabel("Jumlah Klaim")
ax3.set_xticks(range(1, 13))
ax3.legend()

# ---- Plot 3: Komposisi Tipe Perawatan ----
ax4 = fig.add_subplot(gs[1, 0])
care_counts = df["Inpatient/Outpatient"].value_counts()
colors_pie  = [AXA_RED, AXA_DARK, AXA_BLUE, AXA_GRAY, AXA_GREEN]
ax4.pie(care_counts.values, labels=care_counts.index,
        autopct="%1.1f%%", colors=colors_pie[:len(care_counts)],
        startangle=90, pctdistance=0.85)
ax4.set_title("Komposisi Tipe Perawatan", fontweight="bold")

# ---- Plot 4: Rata-rata Klaim per Lokasi RS ----
ax5 = fig.add_subplot(gs[1, 1])
lokasi_avg = df.groupby("Lokasi RS")[TARGET_COL].median().sort_values(ascending=False)
bars = ax5.barh(lokasi_avg.index, lokasi_avg.values,
                color=[AXA_RED if v > lokasi_avg.median() else AXA_GRAY
                       for v in lokasi_avg.values])
ax5.set_title("Median Klaim per Lokasi RS", fontweight="bold")
ax5.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))
ax5.set_xlabel("Median Nominal Klaim (IDR)")

# ---- Plot 5: Top 10 ICD Diagnosis by Total Cost ----
ax6 = fig.add_subplot(gs[1, 2])
top_icd = df.groupby("ICD Description")[TARGET_COL].sum().nlargest(10)
ax6.barh(range(len(top_icd)), top_icd.values, color=AXA_DARK, alpha=0.85)
ax6.set_yticks(range(len(top_icd)))
ax6.set_yticklabels([x[:25] + "..." if len(x) > 25 else x
                     for x in top_icd.index], fontsize=8)
ax6.set_title("Top 10 Diagnosis (Total Biaya Klaim)", fontweight="bold")
ax6.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e9:.1f}B"))
ax6.set_xlabel("Total Klaim (IDR Milyar)")

fig.suptitle("AXA Insurance - Eksplorasi Data Klaim", fontsize=16,
             fontweight="bold", color=AXA_DARK, y=1.01)
plt.savefig("eda_overview.png", bbox_inches="tight", dpi=120)
plt.show()
print("EDA selesai. Temuan kunci:")
print(f"  Median klaim: Rp {df[TARGET_COL].median():,.0f}")
print(f"  Mean klaim  : Rp {df[TARGET_COL].mean():,.0f}")
print(f"  Rasio mean/median (skewness proxy): {df[TARGET_COL].mean()/df[TARGET_COL].median():.2f}x")


## Sel 4 — Core Model 1: Isolation Forest (Anomaly Detection)

Isolation Forest adalah algoritma berbasis pohon keputusan acak yang mengidentifikasi anomali
berdasarkan seberapa "mudah" sebuah titik data diisolasi dari titik lainnya.
Titik anomali (klaim mencurigakan) memerlukan lebih sedikit partisi untuk diisolasi,
sehingga mendapat skor anomali yang tinggi.

Parameter kunci `contamination=0.05` berarti kita mendefinisikan bahwa sekitar 5 persen
dari populasi klaim adalah anomali potensial, selaras dengan target Top 5% Decision Engine.


In [ ]:
# ============================================================
# SEL 4: ISOLATION FOREST — ANOMALY DETECTION
# Logika: Klaim anomali adalah klaim yang secara statistik
# "jauh" dari mayoritas pola klaim normal. Isolation Forest
# mendeteksi ini tanpa memerlukan label fraud eksplisit
# (unsupervised approach), yang ideal karena dataset ini
# tidak memiliki kolom ground-truth "fraud/not-fraud".
# ============================================================

# Pilih fitur yang paling informatif untuk deteksi anomali.
# Kombinasi fitur finansial, temporal, dan perilaku memberikan
# ruang fitur multi-dimensi untuk isolasi yang akurat.
IF_FEATURES = [
    "Nominal Klaim Yang Disetujui_Scaled",
    "Nominal Biaya RS Yang Terjadi_Scaled",
    "Cost_Gap_Scaled",
    "Patient_Age_Scaled",
    "Policy_Tenure_Days_Scaled",
    "Claim_Frequency_Scaled",
    "Treatment_Duration_Scaled",
    "Lokasi_RS_Risk_Score",
    "Is_Inpatient",
    "Overseas_Treatment_Flag",
]

X_if = df[IF_FEATURES].fillna(df[IF_FEATURES].median())

# contamination=0.05 berarti kita mengharapkan sekitar 5% anomali
# n_estimators=200 memberikan stabilitas lebih tinggi dari default (100)
# random_state memastikan reproduktibilitas hasil
iso_forest = IsolationForest(
    n_estimators  = 200,
    contamination = 0.05,
    max_samples   = "auto",
    random_state  = 42,
    n_jobs        = -1
)

iso_forest.fit(X_if)

# score_samples() mengembalikan nilai negatif: semakin negatif, semakin anomali.
# Kita inversi agar skor anomali lebih tinggi = lebih mencurigakan.
df["IF_Anomaly_Score"]  = -iso_forest.score_samples(X_if)
df["IF_Is_Anomaly"]     = (iso_forest.predict(X_if) == -1).astype(int)

n_anomaly = df["IF_Is_Anomaly"].sum()
print(f"Isolation Forest selesai dilatih.")
print(f"  Jumlah klaim terdeteksi sebagai anomali : {n_anomaly} ({n_anomaly/len(df)*100:.1f}%)")
print(f"  Skor anomali rata-rata (anomali)        : {df[df['IF_Is_Anomaly']==1]['IF_Anomaly_Score'].mean():.4f}")
print(f"  Skor anomali rata-rata (normal)         : {df[df['IF_Is_Anomaly']==0]['IF_Anomaly_Score'].mean():.4f}")

# ---- Visualisasi Isolation Forest ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Distribusi skor anomali
axes[0].hist(df[df["IF_Is_Anomaly"]==0]["IF_Anomaly_Score"],
             bins=50, color=AXA_BLUE, alpha=0.7, label="Normal", density=True)
axes[0].hist(df[df["IF_Is_Anomaly"]==1]["IF_Anomaly_Score"],
             bins=30, color=AXA_RED, alpha=0.8, label="Anomali", density=True)
axes[0].set_title("Distribusi Skor Anomali", fontweight="bold")
axes[0].set_xlabel("Anomaly Score (semakin tinggi = semakin anomali)")
axes[0].legend()

# Plot 2: Scatter klaim vs biaya RS, warna anomali
scatter = axes[1].scatter(
    df["Nominal Biaya RS Yang Terjadi"].clip(upper=df["Nominal Biaya RS Yang Terjadi"].quantile(0.99)),
    df["Nominal Klaim Yang Disetujui"].clip(upper=df["Nominal Klaim Yang Disetujui"].quantile(0.99)),
    c=df["IF_Is_Anomaly"], cmap="RdBu_r", alpha=0.4, s=10
)
axes[1].set_title("Biaya RS vs Nominal Klaim (Merah = Anomali)", fontweight="bold")
axes[1].set_xlabel("Nominal Biaya RS (IDR)")
axes[1].set_ylabel("Nominal Klaim Disetujui (IDR)")
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))

# Plot 3: Breakdown anomali per lokasi RS
anomaly_by_lokasi = df.groupby("Lokasi RS")["IF_Is_Anomaly"].mean().sort_values(ascending=False)
bars = axes[2].bar(anomaly_by_lokasi.index, anomaly_by_lokasi.values * 100,
                   color=[AXA_RED if v > 0.05 else AXA_GRAY for v in anomaly_by_lokasi.values])
axes[2].set_title("Tingkat Anomali per Lokasi RS (%)", fontweight="bold")
axes[2].set_xlabel("Lokasi RS")
axes[2].set_ylabel("Persentase Anomali (%)")
axes[2].axhline(5, color=AXA_DARK, linestyle="--", linewidth=1, label="Threshold 5%")
axes[2].legend()
plt.xticks(rotation=30, ha="right")

plt.suptitle("Isolation Forest — Hasil Deteksi Anomali", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("isolation_forest_results.png", bbox_inches="tight", dpi=120)
plt.show()

## Sel 5 — Core Model 2: K-Means Clustering (Risk Profiling)

K-Means mengelompokkan nasabah ke dalam klaster-klaster berdasarkan kemiripan profil klaim mereka.
Berbeda dari Isolation Forest yang mencari outlier, K-Means membangun segmentasi risiko:
mana klaster yang merupakan "nasabah normal", "nasabah berisiko sedang", dan "nasabah berisiko tinggi".

Kita menggunakan metode Elbow untuk menentukan jumlah klaster optimal secara objektif,
daripada menentukan secara arbitrer.


In [ ]:
# ============================================================
# SEL 5: K-MEANS CLUSTERING — RISK SEGMENTATION
# Logika: Kita ingin mengelompokkan nasabah berdasarkan
# perilaku klaim mereka. Klaster dengan nilai rata-rata
# Nominal Klaim, Claim_Frequency, dan Anomaly Score tinggi
# adalah klaster risiko tinggi yang menjadi prioritas
# pengawasan aktuaria.
# ============================================================

# Fitur untuk K-Means: campuran fitur finansial dan perilaku
KM_FEATURES = [
    "Nominal Klaim Yang Disetujui_Scaled",
    "Claim_Frequency_Scaled",
    "Patient_Age_Scaled",
    "Treatment_Duration_Scaled",
    "Cost_Gap_Scaled",
    "Lokasi_RS_Risk_Score",
    "IF_Anomaly_Score",          # Menyertakan skor anomali sebagai input
    "Overseas_Treatment_Flag",
    "Chronic_Disease_Flag",
    "Is_Inpatient",
]

X_km = df[KM_FEATURES].fillna(df[KM_FEATURES].median())

# ---- Elbow Method: menentukan K optimal ----
inertias   = []
sil_scores = []
K_range    = range(2, 10)

print("Menghitung Elbow Method...")
for k in K_range:
    km_test = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels  = km_test.fit_predict(X_km)
    inertias.append(km_test.inertia_)
    sil_scores.append(silhouette_score(X_km, labels, sample_size=2000, random_state=42))
    print(f"  K={k}: Inertia={km_test.inertia_:,.0f} | Silhouette={sil_scores[-1]:.4f}")

# Pilih K optimal berdasarkan Silhouette Score tertinggi
optimal_k = K_range[np.argmax(sil_scores)]
print(f"\nK optimal (Silhouette terbaik): K = {optimal_k}")

# ---- Latih model final dengan K optimal ----
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=20)
df["KM_Cluster"] = kmeans.fit_predict(X_km)

# ---- Beri label klaster berdasarkan rata-rata klaim ----
cluster_stats = df.groupby("KM_Cluster")[TARGET_COL].mean().sort_values(ascending=False)
cluster_rank  = {cid: rank for rank, cid in enumerate(cluster_stats.index)}
df["KM_Risk_Rank"] = df["KM_Cluster"].map(cluster_rank)  # 0 = risiko tertinggi

# ---- Visualisasi K-Means ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Elbow Curve
ax_twin = axes[0].twinx()
axes[0].plot(list(K_range), inertias, "o-", color=AXA_DARK, linewidth=2, label="Inertia")
ax_twin.plot(list(K_range), sil_scores, "s--", color=AXA_RED, linewidth=2, label="Silhouette")
axes[0].axvline(optimal_k, color=AXA_GREEN, linestyle=":", linewidth=2, label=f"K={optimal_k}")
axes[0].set_title("Elbow Method + Silhouette", fontweight="bold")
axes[0].set_xlabel("Jumlah Klaster (K)")
axes[0].set_ylabel("Inertia")
ax_twin.set_ylabel("Silhouette Score", color=AXA_RED)
axes[0].legend(loc="upper left")
ax_twin.legend(loc="upper right")

# Plot 2: Scatter Plot Klaster (PCA 2D projection)
from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_km)
colors_cluster = plt.cm.Set1(np.linspace(0, 1, optimal_k))

for c in range(optimal_k):
    mask = df["KM_Cluster"] == c
    mean_klaim = df[mask][TARGET_COL].mean() / 1e6
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=[colors_cluster[c]], alpha=0.5, s=15,
                    label=f"Klaster {c} (avg Rp{mean_klaim:.0f}M)")

axes[1].set_title(f"K-Means Klaster (PCA 2D) — K={optimal_k}", fontweight="bold")
axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
axes[1].legend(fontsize=7)

# Plot 3: Profil rata-rata klaim per klaster
cluster_profile = df.groupby("KM_Cluster").agg(
    Avg_Klaim      = (TARGET_COL, "mean"),
    Avg_Freq       = ("Claim_Frequency", "mean"),
    Avg_Age        = ("Patient_Age", "mean"),
    Count          = ("Claim ID", "count"),
).sort_values("Avg_Klaim", ascending=False).reset_index()

x_pos = range(len(cluster_profile))
bars  = axes[2].bar(x_pos,
                    cluster_profile["Avg_Klaim"],
                    color=[AXA_RED if i == 0 else (AXA_BLUE if i == 1 else AXA_GRAY)
                           for i in range(len(cluster_profile))],
                    alpha=0.85)
axes[2].set_title("Rata-rata Klaim per Klaster", fontweight="bold")
axes[2].set_xlabel("Klaster (diurutkan dari tertinggi)")
axes[2].set_ylabel("Rata-rata Nominal Klaim (IDR)")
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels([f"K{r['KM_Cluster']}\n(N={r['Count']:,})"
                          for _, r in cluster_profile.iterrows()])
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"Rp{x/1e6:.0f}M"))

for bar, (_, row) in zip(bars, cluster_profile.iterrows()):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                 f"Age:{row['Avg_Age']:.0f}y\nFreq:{row['Avg_Freq']:.1f}",
                 ha="center", fontsize=8)

plt.suptitle("K-Means Clustering — Segmentasi Risiko Nasabah", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("kmeans_results.png", bbox_inches="tight", dpi=120)
plt.show()

print("\nProfil Klaster:")
print(cluster_profile.to_string(index=False))


## Sel 6 — Core Model 3: Ridge Regression (Cost Prediction)

Model regresi memprediksi biaya klaim masa depan berdasarkan profil nasabah.
Kita menggunakan Ridge Regression (L2 regularization) daripada OLS biasa karena dataset memiliki
banyak fitur yang saling berkorelasi (multicollinearity), terutama antara fitur finansial dan fitur yang diturunkan.
Ridge menangani multicollinearity dengan menambahkan penalty pada koefisien yang terlalu besar.

Target variabel dikenakan transformasi logaritmik (log1p) sebelum training karena distribusi
Nominal Klaim sangat positively skewed (CoV 239.8%). Model regresi linier bekerja jauh lebih baik
pada distribusi yang mendekati normal.


In [ ]:
# ============================================================
# SEL 6: RIDGE REGRESSION — COST PREDICTION
# Logika: Dengan memprediksi biaya klaim masa depan,
# tim aktuaria dapat menghitung cadangan klaim (IBNR reserves)
# yang lebih akurat dan menyesuaikan premi nasabah berisiko tinggi.
# ============================================================

# Fitur untuk regresi: gabungan semua fitur yang siap pakai
REG_FEATURES = [
    "Nominal Biaya RS Yang Terjadi_Scaled",
    "Patient_Age_Scaled",
    "Policy_Tenure_Days_Scaled",
    "Claim_Frequency_Scaled",
    "Treatment_Duration_Scaled",
    "Cost_Gap_Scaled",
    "Is_Inpatient",
    "Overseas_Treatment_Flag",
    "Chronic_Disease_Flag",
    "Lokasi_RS_Risk_Score",
    "Gender_Encoded",
    "Care_Type_Encoded",
    "Plan_Code_Encoded",
    "ICD_Category_Encoded",
    "Month_Claim",
    "Quarter_Claim",
    "KM_Risk_Rank",          # Skor klaster dari K-Means
    "IF_Anomaly_Score",      # Skor anomali dari Isolation Forest
]

X_reg = df[REG_FEATURES].fillna(df[REG_FEATURES].median())
y_raw = df[TARGET_COL].fillna(0)

# Log-transform target untuk menormalkan distribusi skewed
y_log = np.log1p(y_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X_reg, y_log, test_size=0.2, random_state=42
)

# Ridge Regression: alpha=10 dipilih sebagai titik awal yang baik
# untuk data dengan multicollinearity tinggi
ridge = Ridge(alpha=10.0)
ridge.fit(X_train, y_train)

# Evaluasi pada test set
y_pred_log = ridge.predict(X_test)
y_pred_raw = np.expm1(y_pred_log)   # Invers log-transform
y_test_raw = np.expm1(y_test)

mae  = mean_absolute_error(y_test_raw, y_pred_raw)
rmse = np.sqrt(mean_squared_error(y_test_raw, y_pred_raw))
r2   = r2_score(y_test_raw, y_pred_raw)
r2_log = r2_score(y_test, y_pred_log)

# Cross-validation untuk estimasi yang lebih robust
cv_scores = cross_val_score(ridge, X_reg, y_log, cv=5, scoring="r2", n_jobs=-1)

print("Ridge Regression — Hasil Evaluasi:")
print(f"  MAE (test set)         : Rp {mae:,.0f}")
print(f"  RMSE (test set)        : Rp {rmse:,.0f}")
print(f"  R² (skala asli)        : {r2:.4f}")
print(f"  R² (log-scale)         : {r2_log:.4f}")
print(f"  R² Cross-Val (5-fold)  : {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

# Prediksi pada seluruh dataset untuk Decision Engine
df["Reg_Predicted_Cost"]  = np.expm1(ridge.predict(X_reg))
df["Reg_Cost_Percentile"] = df["Reg_Predicted_Cost"].rank(pct=True)

# ---- Visualisasi Regresi ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Actual vs Predicted (log scale)
clip_val = np.percentile(y_test_raw, 98)
axes[0].scatter(y_test_raw.clip(upper=clip_val),
                y_pred_raw.clip(upper=clip_val),
                alpha=0.3, s=10, color=AXA_DARK)
max_val = clip_val
axes[0].plot([0, max_val], [0, max_val], "r--", linewidth=2, label="Perfect Fit")
axes[0].set_title(f"Actual vs Predicted (R²={r2:.3f})", fontweight="bold")
axes[0].set_xlabel("Actual Nominal Klaim")
axes[0].set_ylabel("Predicted Nominal Klaim")
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))
axes[0].legend()

# Plot 2: Residual Plot
residuals = y_test_raw - y_pred_raw
axes[1].scatter(y_pred_raw.clip(upper=clip_val), residuals.clip(-2e8, 2e8),
                alpha=0.3, s=10, color=AXA_BLUE)
axes[1].axhline(0, color=AXA_RED, linewidth=2, linestyle="--")
axes[1].set_title("Residual Plot", fontweight="bold")
axes[1].set_xlabel("Predicted Value")
axes[1].set_ylabel("Residual (Actual - Predicted)")

# Plot 3: Feature Importance (koefisien Ridge)
coef_df = pd.DataFrame({
    "Feature": REG_FEATURES,
    "Coefficient": ridge.coef_
}).reindex(pd.Series(ridge.coef_).abs().sort_values(ascending=False).index)
coef_top = coef_df.head(12)

colors_coef = [AXA_RED if c > 0 else AXA_BLUE for c in coef_top["Coefficient"]]
axes[2].barh(range(len(coef_top)), coef_top["Coefficient"],
             color=colors_coef, alpha=0.85)
axes[2].set_yticks(range(len(coef_top)))
axes[2].set_yticklabels(coef_top["Feature"], fontsize=8)
axes[2].axvline(0, color="black", linewidth=1)
axes[2].set_title("Top 12 Feature Coefficients (Ridge)", fontweight="bold")

plt.suptitle("Ridge Regression — Prediksi Biaya Klaim", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("regression_results.png", bbox_inches="tight", dpi=120)
plt.show()


## Sel 7 — Comparator Model 1: Forward/Backward Chaining (Rule-Based Expert System)

Forward Chaining adalah inferensi "data-driven": kita mulai dari fakta yang ada dan
menerapkan aturan secara berurutan hingga mencapai kesimpulan. Backward Chaining adalah
kebalikannya: kita mulai dari hipotesis ("apakah ini nasabah risiko tinggi?") dan
menelusuri balik fakta apa yang mendukungnya.

Sistem ini merepresentasikan pengetahuan underwriter berpengalaman dalam bentuk aturan
IF-THEN yang eksplisit. Ini adalah pendekatan yang mudah diaudit dan dijelaskan kepada
regulator, namun kelemahannya adalah rigid dan tidak dapat menangkap pola kompleks
yang tidak terduga.


In [ ]:
# ============================================================
# SEL 7: FORWARD/BACKWARD CHAINING — RULE-BASED EXPERT SYSTEM
# Logika: Sistem pakar berbasis aturan ini merepresentasikan
# knowledge underwriter AXA dalam bentuk proposisi logika.
# Setiap aturan adalah pengkodean kebijakan bisnis yang eksplisit.
# ============================================================

class AXARuleEngine:
    """
    Mesin inferensi berbasis aturan untuk penilaian risiko klaim.

    Mengimplementasikan dua mode:
    - forward_chain(): Data-driven, semua aturan dievaluasi secara paralel
    - backward_chain(): Goal-driven, aturan dievaluasi untuk membuktikan hipotesis
    """

    def __init__(self, df_ref: pd.DataFrame):
        # Threshold dinamis berbasis distribusi data aktual
        self.p75_klaim  = df_ref["Nominal Klaim Yang Disetujui"].quantile(0.75)
        self.p90_klaim  = df_ref["Nominal Klaim Yang Disetujui"].quantile(0.90)
        self.p75_freq   = df_ref["Claim_Frequency"].quantile(0.75)
        self.p75_dur    = df_ref["Treatment_Duration"].quantile(0.75)
        self.p50_age    = df_ref["Patient_Age"].quantile(0.50)

    def _rule_high_nominal(self, row) -> bool:
        """R1: Nominal klaim di atas P90 adalah indikator utama risiko tinggi."""
        return row["Nominal Klaim Yang Disetujui"] >= self.p90_klaim

    def _rule_high_frequency(self, row) -> bool:
        """R2: Frekuensi klaim di atas P75 mengindikasikan utilisasi berlebih."""
        return row["Claim_Frequency"] >= self.p75_freq

    def _rule_overseas_treatment(self, row) -> bool:
        """R3: Perawatan di luar Indonesia memiliki biaya rata-rata 3-5x lebih tinggi."""
        return row["Overseas_Treatment_Flag"] == 1

    def _rule_chronic_disease(self, row) -> bool:
        """R4: Penyakit kronis menghasilkan pola klaim berulang jangka panjang."""
        return row["Chronic_Disease_Flag"] == 1

    def _rule_long_treatment(self, row) -> bool:
        """R5: Durasi rawat inap panjang (di atas P75) berkorelasi dengan penyakit serius."""
        return row["Treatment_Duration"] >= self.p75_dur

    def _rule_senior_patient(self, row) -> bool:
        """R6: Pasien di atas usia median memiliki risiko penyakit degeneratif lebih tinggi."""
        return row["Patient_Age"] >= self.p50_age

    def _rule_high_cost_gap(self, row) -> bool:
        """R7: Cost gap negatif (klaim > biaya RS) adalah sinyal inkonsistensi data."""
        return row["Cost_Gap"] < 0

    def forward_chain(self, row) -> dict:
        """
        Forward Chaining: evaluasi semua aturan, akumulasi skor.
        Setiap aturan yang terpenuhi menambah skor risiko.
        Bobot berbeda mencerminkan kepentingan relatif setiap indikator.
        """
        facts = {
            "high_nominal"    : self._rule_high_nominal(row),
            "high_frequency"  : self._rule_high_frequency(row),
            "overseas"        : self._rule_overseas_treatment(row),
            "chronic"         : self._rule_chronic_disease(row),
            "long_treatment"  : self._rule_long_treatment(row),
            "senior_patient"  : self._rule_senior_patient(row),
            "cost_gap_anomaly": self._rule_high_cost_gap(row),
        }
        weights = {
            "high_nominal"    : 3.0,  # Indikator terkuat
            "high_frequency"  : 2.5,
            "overseas"        : 2.0,
            "chronic"         : 2.0,
            "long_treatment"  : 1.5,
            "senior_patient"  : 1.0,
            "cost_gap_anomaly": 2.0,
        }
        score = sum(weights[k] for k, v in facts.items() if v)
        max_score = sum(weights.values())
        return {
            "FC_Score"         : score,
            "FC_Normalized"    : score / max_score,
            "FC_Risk_Level"    : "TINGGI" if score >= 6 else ("SEDANG" if score >= 3 else "RENDAH"),
            **{f"FC_Rule_{k}": int(v) for k, v in facts.items()}
        }

    def backward_chain(self, row, hypothesis: str = "high_risk") -> dict:
        """
        Backward Chaining: mulai dari hipotesis 'nasabah risiko tinggi',
        cari fakta-fakta yang mendukung. Mengembalikan True jika hipotesis
        terbukti dari fakta yang tersedia.

        Pohon inferensi untuk hipotesis 'high_risk':
        high_risk <-- (high_nominal AND (overseas OR chronic))
                   OR (high_frequency AND long_treatment)
                   OR (cost_gap_anomaly)
        """
        if hypothesis == "high_risk":
            branch_1 = (self._rule_high_nominal(row) and
                        (self._rule_overseas_treatment(row) or self._rule_chronic_disease(row)))
            branch_2 = (self._rule_high_frequency(row) and
                        self._rule_long_treatment(row))
            branch_3 = self._rule_high_cost_gap(row)
            proved    = branch_1 or branch_2 or branch_3
            return {
                "BC_High_Risk"     : int(proved),
                "BC_Branch1_Proved": int(branch_1),
                "BC_Branch2_Proved": int(branch_2),
                "BC_Branch3_Proved": int(branch_3),
            }
        return {"BC_High_Risk": 0}


# ---- Inisialisasi dan jalankan engine ----
engine = AXARuleEngine(df)

print("Menjalankan Forward dan Backward Chaining...")
fc_results = df.apply(lambda r: engine.forward_chain(r), axis=1, result_type="expand")
bc_results = df.apply(lambda r: engine.backward_chain(r), axis=1, result_type="expand")

df = pd.concat([df, fc_results, bc_results], axis=1)

print(f"Forward Chain selesai.")
print(f"  Distribusi Risk Level:")
print(df["FC_Risk_Level"].value_counts().to_string())
print(f"\nBackward Chain selesai.")
print(f"  High Risk terdeteksi (BC): {df['BC_High_Risk'].sum():,} ({df['BC_High_Risk'].mean()*100:.1f}%)")


## Sel 8 — Comparator Model 2: Teorema Bayes (Probabilistic Inference)

Teorema Bayes menghitung probabilitas posterior bahwa sebuah klaim adalah "risiko tinggi"
berdasarkan kombinasi bukti yang diamati.

Formula: P(H|E) = P(E|H) x P(H) / P(E)

Di mana H adalah hipotesis "klaim risiko tinggi" dan E adalah kumpulan fitur yang diamati.
Kita menggunakan pendekatan Naive Bayes yang mengasumsikan independensi antar bukti,
kemudian mengkombinasikan probabilitas secara sekuensial (log-space untuk stabilitas numerik).


In [ ]:
# ============================================================
# SEL 8: TEOREMA BAYES — PROBABILISTIC RISK SCORING
# Logika: Berbeda dari rule-based, Bayes mengekspresikan
# ketidakpastian sebagai probabilitas. Ini lebih nuanced:
# sebuah klaim bisa memiliki P(high_risk) = 0.73, bukan
# sekadar "tinggi" atau "rendah".
# ============================================================

class BayesianRiskScorer:
    """
    Penilai risiko berbasis Teorema Bayes dengan Naive Bayes assumption.

    Prior P(H) = proporsi klaim yang termasuk "high value" (P90) dalam data.
    Likelihood P(E|H) dihitung empiris dari data: seberapa sering setiap
    indikator muncul pada klaim high-value vs non-high-value.
    """

    def __init__(self, df_train: pd.DataFrame):
        # Prior: proporsi klaim high-value sebagai definisi "risiko tinggi"
        self.prior_high = df_train["High_Value_Claim_Flag"].mean()
        self.prior_low  = 1.0 - self.prior_high

        # Hitung likelihood setiap indikator dari data empiris
        # P(indikator=1 | High Risk) dan P(indikator=1 | Low Risk)
        self.indicators = [
            "Overseas_Treatment_Flag",
            "Chronic_Disease_Flag",
            "Is_Inpatient",
            "New_Policy_Claim_Flag",
        ]

        self.likelihoods = {}
        for ind in self.indicators:
            high_group = df_train[df_train["High_Value_Claim_Flag"] == 1]
            low_group  = df_train[df_train["High_Value_Claim_Flag"] == 0]
            # Laplace smoothing (+1/+2) untuk menghindari likelihood = 0
            p_given_high = (high_group[ind].sum() + 1) / (len(high_group) + 2)
            p_given_low  = (low_group[ind].sum() + 1) / (len(low_group) + 2)
            self.likelihoods[ind] = {"high": p_given_high, "low": p_given_low}

        # Continuous features: gunakan bin-based likelihood
        # Apakah Claim_Frequency di atas median?
        freq_median = df_train["Claim_Frequency"].median()
        high_above  = (df_train[df_train["High_Value_Claim_Flag"]==1]["Claim_Frequency"] >= freq_median).mean()
        low_above   = (df_train[df_train["High_Value_Claim_Flag"]==0]["Claim_Frequency"] >= freq_median).mean()
        self.likelihoods["high_freq"] = {
            "threshold": freq_median,
            "high": max(high_above, 0.01),
            "low" : max(low_above, 0.01)
        }

    def compute_posterior(self, row) -> float:
        """
        Hitung P(High Risk | Evidence) menggunakan log-space Naive Bayes.
        Log-space digunakan untuk stabilitas numerik (menghindari underflow
        ketika mengalikan banyak probabilitas kecil).
        """
        # Mulai dengan log-prior
        log_posterior_high = np.log(self.prior_high + 1e-10)
        log_posterior_low  = np.log(self.prior_low + 1e-10)

        # Update dengan setiap likelihood
        for ind in self.indicators:
            val = row.get(ind, 0)
            lk  = self.likelihoods[ind]
            if val == 1:
                log_posterior_high += np.log(lk["high"] + 1e-10)
                log_posterior_low  += np.log(lk["low"] + 1e-10)
            else:
                log_posterior_high += np.log(1 - lk["high"] + 1e-10)
                log_posterior_low  += np.log(1 - lk["low"] + 1e-10)

        # Continuous: Claim_Frequency
        lk_freq = self.likelihoods["high_freq"]
        if row.get("Claim_Frequency", 0) >= lk_freq["threshold"]:
            log_posterior_high += np.log(lk_freq["high"] + 1e-10)
            log_posterior_low  += np.log(lk_freq["low"] + 1e-10)

        # Normalisasi (log-sum-exp)
        log_max    = max(log_posterior_high, log_posterior_low)
        prob_high  = np.exp(log_posterior_high - log_max)
        prob_low   = np.exp(log_posterior_low  - log_max)
        posterior  = prob_high / (prob_high + prob_low)

        return float(posterior)


bayes_scorer = BayesianRiskScorer(df)

print("Menghitung skor Bayesian untuk semua klaim...")
df["Bayes_Risk_Score"]  = df.apply(
    lambda row: bayes_scorer.compute_posterior(row), axis=1
)
df["Bayes_High_Risk"]   = (df["Bayes_Risk_Score"] >= 0.5).astype(int)

print(f"Bayesian Scoring selesai.")
print(f"  Skor rata-rata (semua klaim)    : {df['Bayes_Risk_Score'].mean():.4f}")
print(f"  Skor rata-rata (high-value klaim): {df[df['High_Value_Claim_Flag']==1]['Bayes_Risk_Score'].mean():.4f}")
print(f"  P(High Risk) prior              : {bayes_scorer.prior_high:.4f}")
print(f"  Likelihoods yang dihitung:")
for key, val in bayes_scorer.likelihoods.items():
    print(f"    {key}: P(.|High)={val['high']:.3f}, P(.|Low)={val['low']:.3f}")


## Sel 9 — Comparator Model 3: Certainty Factors (CF)

Certainty Factors adalah teknik dari sistem pakar MYCIN (Stanford, 1970an) yang menangani
ketidakpastian secara berbeda dari probabilitas. CF merepresentasikan "keyakinan bersih"
(net belief) terhadap suatu hipotesis, berada di rentang [-1, +1].

CF = +1 berarti certainty mutlak hipotesis benar. CF = -1 berarti certainty mutlak hipotesis salah.
CF = 0 berarti tidak ada bukti sama sekali.

Keunggulan CF adalah kemampuannya mengkombinasikan bukti yang saling memperkuat (confirmatory)
maupun saling melemahkan (contradictory) menggunakan formula khusus yang intuitif secara logis.


In [ ]:
# ============================================================
# SEL 9: CERTAINTY FACTORS — EXPERT SYSTEM MYCIN-STYLE
# Logika: CF mengkombinasikan "derajat kepercayaan" terhadap
# setiap aturan. Berbeda dari Bayes yang mengandalkan data
# historis, CF mengandalkan penilaian ekspertis (bobot CF
# setiap aturan idealnya dikalibrasi dengan underwriter AXA).
# ============================================================

def combine_cf(cf1: float, cf2: float) -> float:
    """
    Menggabungkan dua Certainty Factor menggunakan formula MYCIN standar.

    Kasus 1: Dua bukti yang sama-sama mendukung hipotesis (positif)
             CF_combined = CF1 + CF2 * (1 - CF1)
    Kasus 2: Dua bukti yang sama-sama menentang hipotesis (negatif)
             CF_combined = CF1 + CF2 * (1 + CF1)
    Kasus 3: Satu bukti mendukung, satu menentang
             CF_combined = (CF1 + CF2) / (1 - min(|CF1|, |CF2|))
    """
    if cf1 >= 0 and cf2 >= 0:
        return cf1 + cf2 * (1 - cf1)
    elif cf1 < 0 and cf2 < 0:
        return cf1 + cf2 * (1 + cf1)
    else:
        denom = 1 - min(abs(cf1), abs(cf2))
        return (cf1 + cf2) / denom if denom != 0 else 0.0


def compute_cf_score(row, p75_klaim: float, p75_freq: float,
                     p75_dur: float, p50_age: float) -> float:
    """
    Menghitung CF kumulatif untuk satu klaim.

    Setiap aturan memiliki CF yang menunjukkan derajat kepercayaan
    bahwa jika aturan ini terpenuhi, nasabah adalah risiko tinggi.
    Nilai CF ini idealnya dikalibrasi oleh underwriter berpengalaman.
    """
    cf_evidence = []

    # Aturan 1: Nominal klaim tinggi (CF = 0.8: bukti sangat kuat)
    if row["Nominal Klaim Yang Disetujui"] >= p75_klaim:
        cf_evidence.append(0.80)

    # Aturan 2: Frekuensi klaim tinggi (CF = 0.70)
    if row["Claim_Frequency"] >= p75_freq:
        cf_evidence.append(0.70)

    # Aturan 3: Perawatan overseas (CF = 0.65)
    if row["Overseas_Treatment_Flag"] == 1:
        cf_evidence.append(0.65)

    # Aturan 4: Penyakit kronis (CF = 0.60)
    if row["Chronic_Disease_Flag"] == 1:
        cf_evidence.append(0.60)

    # Aturan 5: Durasi rawat panjang (CF = 0.50)
    if row["Treatment_Duration"] >= p75_dur:
        cf_evidence.append(0.50)

    # Aturan 6: Usia senior (CF = 0.35: bukti moderat)
    if row["Patient_Age"] >= p50_age:
        cf_evidence.append(0.35)

    # Aturan PENGURANG: Polis lama dengan tenure panjang (CF = -0.30)
    # Nasabah loyal jangka panjang sedikit lebih kecil risikonya
    if row["Policy_Tenure_Days"] > 1825:  # 5 tahun
        cf_evidence.append(-0.30)

    # Aturan PENGURANG: Klaim domestik dengan nilai rendah (CF = -0.25)
    if row["Overseas_Treatment_Flag"] == 0 and        row["Nominal Klaim Yang Disetujui"] < row["Nominal Klaim Yang Disetujui"]:
        cf_evidence.append(-0.20)

    if not cf_evidence:
        return 0.0

    # Akumulasi semua CF secara sekuensial
    combined = cf_evidence[0]
    for cf_next in cf_evidence[1:]:
        combined = combine_cf(combined, cf_next)

    # Klip hasil ke [-1, 1]
    return float(np.clip(combined, -1.0, 1.0))


# Hitung parameter threshold
p75_klaim = df["Nominal Klaim Yang Disetujui"].quantile(0.75)
p75_freq  = df["Claim_Frequency"].quantile(0.75)
p75_dur   = df["Treatment_Duration"].quantile(0.75)
p50_age   = df["Patient_Age"].quantile(0.50)

print("Menghitung Certainty Factors untuk semua klaim...")
df["CF_Score"] = df.apply(
    lambda row: compute_cf_score(row, p75_klaim, p75_freq, p75_dur, p50_age),
    axis=1
)
df["CF_High_Risk"]    = (df["CF_Score"] >= 0.50).astype(int)
df["CF_Normalized"]   = (df["CF_Score"] + 1) / 2   # Normalisasi ke [0,1]

print(f"Certainty Factors selesai.")
print(f"  CF rata-rata              : {df['CF_Score'].mean():.4f}")
print(f"  CF >= 0.5 (High Risk)     : {df['CF_High_Risk'].sum():,} ({df['CF_High_Risk'].mean()*100:.1f}%)")
print(f"  CF maksimum               : {df['CF_Score'].max():.4f}")
print(f"  CF minimum                : {df['CF_Score'].min():.4f}")


## Sel 10 — Evaluasi dan Perbandingan Semua Model

Pada tahap ini kita membandingkan hasil keenam model secara berdampingan.
Karena dataset tidak memiliki label ground-truth "fraud/berisiko", kita menggunakan
`High_Value_Claim_Flag` (klaim di atas P90) sebagai proxy variabel target evaluasi.
Ini bukan evaluasi sempurna, namun cukup representatif untuk membandingkan kemampuan
setiap model dalam mengidentifikasi klaim bernilai tinggi yang memerlukan perhatian khusus.


In [ ]:
# ============================================================
# SEL 10: EVALUASI KOMPARATIF SEMUA MODEL
# Logika: Menggunakan High_Value_Claim_Flag (P90) sebagai proxy
# ground truth. Setiap model di-evaluate pada kemampuannya
# mengidentifikasi klaim high-value ini.
# ============================================================

y_true = df["High_Value_Claim_Flag"].values

# Konversi setiap model ke prediksi biner (0/1) dan skor probabilistik
model_predictions = {
    "Isolation Forest"    : df["IF_Is_Anomaly"].values,
    "K-Means (Klaster 0)" : (df["KM_Risk_Rank"] == 0).astype(int).values,
    "Ridge Regression"    : (df["Reg_Cost_Percentile"] >= 0.90).astype(int).values,
    "Forward Chaining"    : (df["FC_Risk_Level"] == "TINGGI").astype(int).values,
    "Backward Chaining"   : df["BC_High_Risk"].values,
    "Teorema Bayes"       : df["Bayes_High_Risk"].values,
    "Certainty Factors"   : df["CF_High_Risk"].values,
}

model_scores = {
    "Isolation Forest"    : df["IF_Anomaly_Score"].values,
    "Ridge Regression"    : df["Reg_Cost_Percentile"].values,
    "Forward Chaining"    : df["FC_Normalized"].values,
    "Teorema Bayes"       : df["Bayes_Risk_Score"].values,
    "Certainty Factors"   : df["CF_Normalized"].values,
}

# ---- Tabel Perbandingan Metrik ----
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

results = []
for name, preds in model_predictions.items():
    acc  = accuracy_score(y_true, preds)
    prec = precision_score(y_true, preds, zero_division=0)
    rec  = recall_score(y_true, preds, zero_division=0)
    f1   = f1_score(y_true, preds, zero_division=0)

    # AUC hanya untuk model dengan skor kontinu
    auc = None
    if name in model_scores:
        try:
            auc = roc_auc_score(y_true, model_scores[name])
        except Exception:
            auc = None

    results.append({
        "Model"     : name,
        "Type"      : "Core ML" if name in ["Isolation Forest", "K-Means (Klaster 0)", "Ridge Regression"] else "Comparator",
        "Accuracy"  : acc,
        "Precision" : prec,
        "Recall"    : rec,
        "F1-Score"  : f1,
        "AUC-ROC"   : auc if auc is not None else "N/A",
    })

results_df = pd.DataFrame(results)
print("=" * 70)
print("TABEL PERBANDINGAN PERFORMA MODEL")
print("=" * 70)
print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))

# ---- Visualisasi Perbandingan ----
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: F1 Score Comparison
core_models  = results_df[results_df["Type"] == "Core ML"]
comp_models  = results_df[results_df["Type"] == "Comparator"]
all_names    = results_df["Model"].tolist()
f1_values    = [r["F1-Score"] for _, r in results_df.iterrows()]
colors_bar   = [AXA_RED if t == "Core ML" else AXA_BLUE
                for t in results_df["Type"]]

axes[0,0].barh(all_names, f1_values, color=colors_bar, alpha=0.85)
axes[0,0].set_title("Perbandingan F1-Score (P90 Ground Truth)", fontweight="bold")
axes[0,0].set_xlabel("F1-Score")
axes[0,0].axvline(results_df["F1-Score"].mean(), color="black",
                  linestyle="--", linewidth=1.5, label="Rata-rata")
axes[0,0].legend()
red_patch  = mpatches.Patch(color=AXA_RED,  label="Core ML")
blue_patch = mpatches.Patch(color=AXA_BLUE, label="Comparator")
axes[0,0].legend(handles=[red_patch, blue_patch], loc="lower right")

# Plot 2: Radar Chart (Precision vs Recall vs F1)
metrics_cols = ["Precision", "Recall", "F1-Score"]
x_pos = np.arange(len(metrics_cols))
width = 0.12
for i, (_, row) in enumerate(results_df.iterrows()):
    vals = [row["Precision"], row["Recall"], row["F1-Score"]]
    axes[0,1].bar(x_pos + i*width, vals, width,
                  label=row["Model"][:18], alpha=0.8)
axes[0,1].set_title("Precision vs Recall vs F1 per Model", fontweight="bold")
axes[0,1].set_xticks(x_pos + width*3)
axes[0,1].set_xticklabels(metrics_cols)
axes[0,1].legend(fontsize=6, ncol=2)

# Plot 3: AUC Comparison (hanya untuk model dengan skor kontinu)
auc_models  = results_df[results_df["AUC-ROC"] != "N/A"].copy()
auc_models["AUC-ROC"] = auc_models["AUC-ROC"].astype(float)
colors_auc  = [AXA_RED if t == "Core ML" else AXA_BLUE
               for t in auc_models["Type"]]
axes[1,0].bar(auc_models["Model"], auc_models["AUC-ROC"],
              color=colors_auc, alpha=0.85)
axes[1,0].axhline(0.5, color="black", linestyle="--", linewidth=1.5,
                  label="Random Classifier (AUC=0.5)")
axes[1,0].set_title("AUC-ROC Comparison", fontweight="bold")
axes[1,0].set_ylabel("AUC-ROC Score")
axes[1,0].tick_params(axis="x", rotation=20)
axes[1,0].set_ylim(0, 1.05)
axes[1,0].legend()

# Plot 4: Distribution of Scores
axes[1,1].hist(df["FC_Normalized"], bins=30, alpha=0.6, label="Forward Chaining", color=AXA_BLUE)
axes[1,1].hist(df["Bayes_Risk_Score"], bins=30, alpha=0.6, label="Bayes", color=AXA_GREEN)
axes[1,1].hist(df["CF_Normalized"], bins=30, alpha=0.6, label="CF", color=AXA_GRAY)
axes[1,1].hist(df["Reg_Cost_Percentile"], bins=30, alpha=0.5, label="Regression (Pctile)", color=AXA_RED)
axes[1,1].set_title("Distribusi Skor Risk per Model", fontweight="bold")
axes[1,1].set_xlabel("Normalized Risk Score [0-1]")
axes[1,1].set_ylabel("Frekuensi")
axes[1,1].legend(fontsize=9)

fig.suptitle("Evaluasi Komparatif: Core ML vs Comparator Models\n(Ground Truth: High Value Claim P90)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("model_comparison.png", bbox_inches="tight", dpi=120)
plt.show()


## Sel 11 — Decision Engine: Dynamic Percentile Ranking

Ini adalah tahap akhir yang mengintegrasikan output dari semua model menjadi satu skor risiko
komposit untuk setiap nasabah. Dynamic Percentile Ranking digunakan sebagai mekanisme keputusan
final karena sifatnya yang adaptif: definisi "nasabah kritis" secara otomatis bergeser mengikuti
distribusi data terbaru, tanpa perlu menyesuaikan threshold secara manual setiap periode.

Pendekatan ini jauh lebih robust dibandingkan threshold statis (misalnya "klaim di atas Rp 100 juta")
yang akan menjadi usang seiring inflasi biaya kesehatan atau perubahan komposisi portofolio.


In [ ]:
# ============================================================
# SEL 11: DECISION ENGINE — DYNAMIC PERCENTILE RANKING
# Logika: Skor komposit adalah rata-rata tertimbang dari
# output semua model, dinormalisasi ke [0,1]. Bobot mencerminkan
# kepercayaan relatif terhadap masing-masing model berdasarkan
# performa evaluasi di Sel 10.
# ============================================================

# Normalisasi IF_Anomaly_Score ke [0,1]
if_min = df["IF_Anomaly_Score"].min()
if_max = df["IF_Anomaly_Score"].max()
df["IF_Score_Normalized"] = (df["IF_Anomaly_Score"] - if_min) / (if_max - if_min + 1e-10)

# Normalisasi KM_Risk_Rank ke [0,1] (0 = risiko tertinggi -> 1.0)
km_max_rank = df["KM_Risk_Rank"].max()
df["KM_Score_Normalized"] = 1.0 - (df["KM_Risk_Rank"] / (km_max_rank + 1e-10))

# ---- Bobot setiap model (total = 1.0) ----
# Core Models mendapat bobot lebih besar karena berbasis data multi-dimensi.
# Comparator Models berfungsi sebagai penguatan (confirmatory evidence).
WEIGHTS = {
    "IF"    : 0.25,   # Isolation Forest: andalan deteksi anomali multivariat
    "KM"    : 0.20,   # K-Means: konteks klaster komunal
    "REG"   : 0.25,   # Regression: prediksi finansial kuantitatif
    "FC"    : 0.10,   # Forward Chaining: domain knowledge underwriter
    "BAYES" : 0.12,   # Bayes: probabilistic evidence integration
    "CF"    : 0.08,   # CF: expert certainty scoring
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9, "Jumlah bobot harus = 1.0"

# ---- Hitung Composite Risk Score ----
df["Composite_Risk_Score"] = (
    WEIGHTS["IF"]    * df["IF_Score_Normalized"]    +
    WEIGHTS["KM"]    * df["KM_Score_Normalized"]    +
    WEIGHTS["REG"]   * df["Reg_Cost_Percentile"]    +
    WEIGHTS["FC"]    * df["FC_Normalized"]           +
    WEIGHTS["BAYES"] * df["Bayes_Risk_Score"]        +
    WEIGHTS["CF"]    * df["CF_Normalized"]
)

# ---- Dynamic Percentile Ranking ----
# Setiap klaim mendapat peringkat persentil berdasarkan skor komposit.
# Klaim di atas P95 adalah "Kritis" (Top 5%).
df["Risk_Percentile_Rank"] = df["Composite_Risk_Score"].rank(pct=True)

THRESHOLD_TOP5   = df["Composite_Risk_Score"].quantile(0.95)
THRESHOLD_TOP20  = df["Composite_Risk_Score"].quantile(0.80)

df["Risk_Category"] = "RENDAH"
df.loc[df["Composite_Risk_Score"] >= THRESHOLD_TOP20, "Risk_Category"] = "SEDANG"
df.loc[df["Composite_Risk_Score"] >= THRESHOLD_TOP5,  "Risk_Category"] = "KRITIS"

risk_dist = df["Risk_Category"].value_counts()
print("=" * 55)
print("DISTRIBUSI KATEGORI RISIKO FINAL")
print("=" * 55)
print(f"  KRITIS (Top 5%)  : {risk_dist.get('KRITIS', 0):>5,} klaim")
print(f"  SEDANG (Top 20%) : {risk_dist.get('SEDANG', 0):>5,} klaim")
print(f"  RENDAH           : {risk_dist.get('RENDAH', 0):>5,} klaim")
print(f"  TOTAL            : {len(df):>5,} klaim")
print()
print(f"  Threshold KRITIS (P95) : {THRESHOLD_TOP5:.4f}")
print(f"  Threshold SEDANG (P80) : {THRESHOLD_TOP20:.4f}")


## Sel 12 — Output Final: Daftar Nasabah Kritis (Top 5%)

Hasil akhir dari seluruh pipeline ini adalah daftar nasabah yang masuk kategori KRITIS.
Daftar ini dilengkapi dengan breakdown skor per model agar tim aktuaria dapat memahami
"mengapa" seorang nasabah dikategorikan kritis, bukan hanya "siapa" yang kritis.
Transparansi ini (explainability) adalah komponen kritis dalam pengambilan keputusan
di industri asuransi yang diregulasi.


In [ ]:
# ============================================================
# SEL 12: OUTPUT NASABAH KRITIS — TOP 5% RISK LIST
# ============================================================

# ---- Agregasi ke level nasabah (Nomor Polis) ----
# Seorang nasabah bisa memiliki banyak klaim.
# Kita ambil nilai maksimum skor komposit di antara semua klaimnya.
nasabah_kritis = (
    df[df["Risk_Category"] == "KRITIS"]
    .sort_values("Composite_Risk_Score", ascending=False)
    .groupby("Nomor Polis")
    .agg(
        Max_Composite_Score = ("Composite_Risk_Score", "max"),
        Max_Percentile_Rank = ("Risk_Percentile_Rank", "max"),
        Total_Klaim_Disetujui = ("Nominal Klaim Yang Disetujui", "sum"),
        Jumlah_Klaim        = ("Claim ID", "count"),
        Avg_IF_Score        = ("IF_Score_Normalized", "mean"),
        Avg_KM_Score        = ("KM_Score_Normalized", "mean"),
        Avg_Reg_Pctile      = ("Reg_Cost_Percentile", "mean"),
        Avg_Bayes_Score     = ("Bayes_Risk_Score", "mean"),
        Avg_CF_Score        = ("CF_Normalized", "mean"),
        Top_Diagnosis       = ("ICD Description", lambda x: x.mode()[0] if len(x) > 0 else "N/A"),
        Lokasi_RS           = ("Lokasi RS", lambda x: x.mode()[0] if len(x) > 0 else "N/A"),
        Plan_Code           = ("Plan Code", "first"),
        Gender              = ("Gender", "first"),
        Patient_Age         = ("Patient_Age", "first"),
    )
    .sort_values("Max_Composite_Score", ascending=False)
    .reset_index()
)

# Tambahkan ranking
nasabah_kritis.insert(0, "Ranking_Risiko", range(1, len(nasabah_kritis) + 1))

print(f"DAFTAR NASABAH KRITIS (TOP 5% RISK)")
print(f"Total nasabah kritis teridentifikasi: {len(nasabah_kritis):,}")
print()

# Tampilkan Top 20 dengan kolom ringkas
display_cols = [
    "Ranking_Risiko", "Nomor Polis", "Max_Composite_Score",
    "Total_Klaim_Disetujui", "Jumlah_Klaim", "Patient_Age",
    "Top_Diagnosis", "Lokasi_RS", "Plan_Code"
]
top20 = nasabah_kritis[display_cols].head(20)

# Format tampilan
top20_display = top20.copy()
top20_display["Max_Composite_Score"]    = top20_display["Max_Composite_Score"].map("{:.4f}".format)
top20_display["Total_Klaim_Disetujui"]  = top20_display["Total_Klaim_Disetujui"].map("Rp{:,.0f}".format)
top20_display["Top_Diagnosis"]          = top20_display["Top_Diagnosis"].str[:30]

print(top20_display.to_string(index=False))


## Sel 13 — Visualisasi Decision Engine dan Summary Dashboard

In [ ]:
# ============================================================
# SEL 13: DASHBOARD VISUALISASI DECISION ENGINE
# ============================================================

fig = plt.figure(figsize=(20, 14))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# ---- Plot 1: Distribusi Composite Risk Score ----
ax1 = fig.add_subplot(gs[0, :2])
ax1.hist(df["Composite_Risk_Score"], bins=80, color=AXA_DARK, alpha=0.7, label="Semua Klaim")
threshold_val = df["Composite_Risk_Score"].quantile(0.95)
ax1.axvline(threshold_val, color=AXA_RED, linewidth=2.5,
            linestyle="--", label=f"P95 Threshold = {threshold_val:.3f}")
ax1.fill_betweenx(
    [0, ax1.get_ylim()[1] if ax1.get_ylim()[1] > 0 else 500],
    threshold_val, df["Composite_Risk_Score"].max(),
    alpha=0.2, color=AXA_RED, label="Zona KRITIS (Top 5%)"
)
ax1.set_title("Distribusi Composite Risk Score (Dynamic Percentile)", fontweight="bold")
ax1.set_xlabel("Composite Risk Score")
ax1.set_ylabel("Frekuensi Klaim")
ax1.legend()

# ---- Plot 2: Risk Category Pie ----
ax2 = fig.add_subplot(gs[0, 2])
cat_counts = df["Risk_Category"].value_counts()
colors_pie  = [AXA_RED, AXA_BLUE, AXA_GRAY]
wedges, texts, autotexts = ax2.pie(
    cat_counts.values, labels=cat_counts.index,
    autopct="%1.1f%%", colors=colors_pie,
    startangle=90, pctdistance=0.75,
    wedgeprops={"edgecolor": "white", "linewidth": 2}
)
ax2.set_title("Distribusi Kategori Risiko", fontweight="bold")

# ---- Plot 3: Score Contribution Heatmap (Top 30 kritis) ----
ax3 = fig.add_subplot(gs[1, :])
score_cols = ["IF_Score_Normalized", "KM_Score_Normalized", "Reg_Cost_Percentile",
              "FC_Normalized", "Bayes_Risk_Score", "CF_Normalized", "Composite_Risk_Score"]
top30_kritis = df[df["Risk_Category"] == "KRITIS"].nlargest(30, "Composite_Risk_Score")
heatmap_data = top30_kritis[score_cols].T
heatmap_data.columns = [f"#{i+1}" for i in range(len(heatmap_data.columns))]

im = ax3.imshow(heatmap_data.values, cmap="YlOrRd", aspect="auto", vmin=0, vmax=1)
ax3.set_yticks(range(len(score_cols)))
ax3.set_yticklabels(["IF Score", "K-Means", "Regression", "Fwd Chain",
                      "Bayes", "CF Score", "KOMPOSIT"], fontsize=9)
ax3.set_xticks(range(len(heatmap_data.columns)))
ax3.set_xticklabels(heatmap_data.columns, fontsize=8, rotation=45)
ax3.set_title("Heatmap Skor Per Model — Top 30 Nasabah Paling Kritis", fontweight="bold")
plt.colorbar(im, ax=ax3, label="Risk Score [0-1]", shrink=0.8)

# ---- Plot 4: Total Eksposur Finansial per Kategori ----
ax4 = fig.add_subplot(gs[2, 0])
eksposur = df.groupby("Risk_Category")[TARGET_COL].sum()
bars4 = ax4.bar(eksposur.index, eksposur.values / 1e9,
                color=[AXA_RED if x == "KRITIS" else (AXA_BLUE if x == "SEDANG" else AXA_GRAY)
                       for x in eksposur.index],
                alpha=0.85)
for bar, val in zip(bars4, eksposur.values):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
             f"Rp{val/1e9:.1f}B", ha="center", fontsize=9, fontweight="bold")
ax4.set_title("Total Eksposur Finansial per Kategori Risiko", fontweight="bold")
ax4.set_ylabel("Total Klaim (IDR Milyar)")

# ---- Plot 5: Top Diagnosis Nasabah Kritis ----
ax5 = fig.add_subplot(gs[2, 1])
top_dx_kritis = (df[df["Risk_Category"] == "KRITIS"]["ICD Description"]
                 .value_counts().head(8))
ax5.barh(range(len(top_dx_kritis)), top_dx_kritis.values, color=AXA_RED, alpha=0.8)
ax5.set_yticks(range(len(top_dx_kritis)))
ax5.set_yticklabels([x[:25] + "..." if len(x) > 25 else x
                     for x in top_dx_kritis.index], fontsize=8)
ax5.set_title("Top Diagnosis Nasabah KRITIS", fontweight="bold")
ax5.set_xlabel("Jumlah Klaim")

# ---- Plot 6: Model Weight Donut ----
ax6 = fig.add_subplot(gs[2, 2])
weight_labels = ["Isolation Forest", "K-Means", "Regression",
                 "Fwd Chaining", "Bayes", "CF"]
weight_values = [WEIGHTS["IF"], WEIGHTS["KM"], WEIGHTS["REG"],
                 WEIGHTS["FC"], WEIGHTS["BAYES"], WEIGHTS["CF"]]
colors_donut  = [AXA_RED, "#E85555", "#FF8C8C", AXA_BLUE, AXA_GREEN, AXA_GRAY]
wedges, texts, autotexts = ax6.pie(
    weight_values, labels=weight_labels, autopct="%1.0f%%",
    colors=colors_donut, startangle=90, pctdistance=0.75,
    wedgeprops={"edgecolor": "white", "linewidth": 2}
)
for at in autotexts:
    at.set_fontsize(8)
ax6.set_title("Komposisi Bobot Composite Score", fontweight="bold")

fig.suptitle("AXA Insurance SPK — Decision Engine Dashboard Top 5% Critical Risk Identification",
             fontsize=16, fontweight="bold", color=AXA_DARK)
plt.savefig("decision_engine_dashboard.png", bbox_inches="tight", dpi=120)
plt.show()

# ---- Summary Bisnis ----
total_klaim_kritis = df[df["Risk_Category"] == "KRITIS"][TARGET_COL].sum()
total_klaim_all    = df[TARGET_COL].sum()

print("=" * 60)
print(" RINGKASAN EKSEKUTIF SPK AXA INSURANCE")
print("=" * 60)
print(f"  Total klaim dianalisis          : {len(df):,}")
print(f"  Total eksposur finansial        : Rp {total_klaim_all/1e9:.2f} Miliar")
print()
print(f"  NASABAH KRITIS (Top 5%)")
print(f"  Jumlah klaim kritis             : {risk_dist.get('KRITIS', 0):,}")
print(f"  Total nilai klaim kritis        : Rp {total_kritis/1e9:.2f} Miliar" if (total_kritis := df[df['Risk_Category']=='KRITIS'][TARGET_COL].sum()) else "")
print(f"  Porsi finansial nasabah kritis  : {total_klaim_kritis/total_klaim_all*100:.1f}% dari total eksposur")
print()
print(f"  REKOMENDASI AKSI:")
print(f"  1. Audit mendalam pada {risk_dist.get('KRITIS', 0)} klaim dalam kategori KRITIS")
print(f"  2. Review ulang premi untuk {nasabah_kritis['Nomor Polis'].nunique()} nasabah kritis unik")
print(f"  3. Fokus verifikasi pada klaim overseas dan penyakit kronis")
print(f"  4. Jalankan ulang model setiap bulan untuk update Dynamic Percentile")


## Sel 14 — Simpan Output Final

In [ ]:
# ============================================================
# SEL 14: SIMPAN HASIL MODELING KE FILE
# ============================================================

# Simpan dataset lengkap dengan semua skor model
output_full_path = "AXA_SPK_Model_Results.csv"
df.to_csv(output_full_path, index=False)
print(f"Dataset hasil modeling disimpan: {output_full_path}")
print(f"  Jumlah baris  : {len(df):,}")
print(f"  Jumlah kolom  : {df.shape[1]}")

# Simpan daftar nasabah kritis ke file terpisah
output_kritis_path = "AXA_Critical_Risk_Policyholders.csv"
nasabah_kritis.to_csv(output_kritis_path, index=False)
print(f"\nDaftar nasabah kritis disimpan : {output_kritis_path}")
print(f"  Total nasabah kritis unik     : {len(nasabah_kritis):,}")

print("\nSemua file berhasil disimpan.")
print("Pipeline SPK AXA Insurance selesai.")
